In [1]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

# This will create a default Spark session
spark = DataprocSparkSession.builder.getOrCreate()
# If you would like to customize the Spark session, please refer to the
# documentation at https://cloud.google.com/bigquery/docs/use-spark
#
# For example, if you need to use a different subnetwork, use the code below

# from google.cloud.dataproc_v1 import Session

# session = Session()
# session.environment_config.execution_config.subnetwork_uri = "<subnetwork_name>"
# spark = DataprocSparkSession.builder.dataprocSessionConfig(session).getOrCreate()

████████████████████████▌                                                       

In [7]:
from google.cloud import bigquery

# Inicializa o cliente do BigQuery padrão
client = bigquery.Client()

# Recupera 10% da base para uma exploração inicial
query = """
    SELECT *
    FROM `project-f1079166-4703-4cc3-975.raw_data.raw_vehicles` TABLESAMPLE SYSTEM (10 PERCENT)
"""

# Traz para um DataFrame do Pandas para inspecionar de forma rápida
df_vehicles = client.query(query).to_dataframe()
display(df_vehicles.head())

,id_sinistro,id_veiculo,marca_modelo,ano_fab,ano_modelo,cor_veiculo,tipo_veiculo,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,ano_mes_sinistro
0,2501575,3082610,None,<NA>,<NA>,None,AUTOMOVEL,2014-12-21,2014,12,21,2014/12
1,2456933,2281029,None,<NA>,<NA>,None,OUTROS,2014-12-23,2014,12,23,2014/12
2,2456933,2280796,GM/VECTRA GLS,1994,1995,Azul,AUTOMOVEL,2014-12-23,2014,12,23,2014/12
3,2456933,1328,I/GM CAPTIVA SPORT V6AWD,2011,2011,Prata,AUTOMOVEL,2014-12-23,2014,12,23,2014/12
4,2463759,12233,HONDA/CIVIC EXS FLEX,2008,2008,Prata,AUTOMOVEL,2014-12-26,2014,12,26,2014/12


In [8]:
# Mostra o tipo de dado de cada coluna e se há valores nulos
print(df_vehicles.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672853 entries, 0 to 672852
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id_sinistro       672853 non-null  Int64 
 1   id_veiculo        672853 non-null  Int64 
 2   marca_modelo      611754 non-null  object
 3   ano_fab           536140 non-null  Int64 
 4   ano_modelo        528722 non-null  Int64 
 5   cor_veiculo       540169 non-null  object
 6   tipo_veiculo      672853 non-null  object
 7   data_sinistro     672853 non-null  dbdate
 8   ano_sinistro      672853 non-null  Int64 
 9   mes_sinistro      672853 non-null  Int64 
 10  dia_sinistro      672853 non-null  Int64 
 11  ano_mes_sinistro  672853 non-null  object
dtypes: Int64(7), dbdate(1), object(4)
memory usage: 66.1+ MB
None



**Resumo:**

* As seguintes colunas estão totalmente preenchidas o que garante uma excelente consistência estrutural: id_sinistro, id_veiculo, data_sinistro, ano_sinistro, mes_sinistro, dia_sinistro, ano_mes_sinistro, tipo_veiculo;
* A tabela de veículos apresenta lacunas em atributos descritivos que exigirão tratamento na camada Silver: marca_modelo, cor_modelo, ano_fab e ano_modelo apresentam  valores nulos e provavel alta cardinalidade.



## Avaliação de Variáveis Categóricas

In [9]:
# Filtra apenas as colunas do tipo objeto (texto/categóricas)
colunas_categoricas = df_vehicles.select_dtypes(include=['object']).columns

# Itera sobre cada coluna categórica mostrando a quantidade e os valores únicos
for coluna in colunas_categoricas:
    print(f"\n==================================================")
    print(f"COLUNA: {coluna}")
    print(f"Total de valores únicos: {df_vehicles[coluna].nunique()}")
    print(f"Valores e frequências (Top 10):\n{df_vehicles[coluna].value_counts().head(10)}")


COLUNA: marca_modelo
Total de valores únicos: 17484
Valores e frequências (Top 10):
marca_modelo
HONDA/CG 150 TITAN KS     12292
HONDA/CG 160 FAN          10786
HONDA/CG 160 START         8965
HONDA/CG 150 FAN ESI       8092
HONDA/BIZ 125 ES           7712
HONDA/CG150 FAN ESDI       7652
HONDA/CBX 250 TWISTER      7181
HONDA/CG 125 FAN KS        7139
HONDA/CB 300R              7019
HONDA/CG 150 TITAN ESD     6352
Name: count, dtype: int64

COLUNA: cor_veiculo
Total de valores únicos: 45
Valores e frequências (Top 10):
cor_veiculo
PRETA       127610
BRANCA       95883
PRATA        91298
VERMELHA     72427
CINZA        49875
AZUL         32992
VERDE        11499
Branco        9763
Preta         8536
Prata         6696
Name: count, dtype: int64

COLUNA: tipo_veiculo
Total de valores únicos: 7
Valores e frequências (Top 10):
tipo_veiculo
AUTOMOVEL         280014
MOTOCICLETA       233982
NAO DISPONIVEL     84091
CAMINHAO           27328
OUTROS             18115
BICICLETA          15175
ONI

**Resumo:**

* marca_modelo possui alta cardinalidade, o Top 10 é 100% dominado por motocicletas da marca Honda. Para a modelagem, o ideal é extrair a cilindrada ou criar uma macro-categoria para os automóveis.
* cor_veiculo apresenta a mesma informação escrito de formas distintas, deverá ser feito um tratamento para unificação dos dados.
* tipo_veiculo: padronizar o texto e garantir que os nulos/indisponíveis sejam tratados adequadamente para não distorcer as métricas no Power BI.



In [10]:
# Resumo estatístico dos anos de fabricação e modelo
print(df_vehicles[['ano_fab', 'ano_modelo']].describe())

          ano_fab   ano_modelo
count    536140.0     528722.0
mean   2009.79253  2010.286213
std      7.847459     7.892666
min        1921.0       1921.0
25%        2007.0       2007.0
50%        2011.0       2012.0
75%        2015.0       2016.0
max        2024.0       2022.0


**Resumo:**

* A grande maioria dos veículos envolvidos em acidentes tem uma idade considerável (cerca de 10 a 15 anos de uso em relação ao período dos registros).
* As métricas de dispersão (std de ~7.8 anos) são muito parecidas entre fabricação e modelo, o que demonstra que a base possui alta consistência lógica entre essas duas variáveis.